In [1]:
import pandas as pd
import numpy as np

from newsapi import NewsApiClient
from transformers import pipeline

In [2]:
import os
from dotenv import load_dotenv
NEWS_API_KEY = os.getenv("NEW_API_KEY") 
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

In [3]:
COMPANY_NAMES = {
    "HDFC BANK": "HDFC Bank",
    "TCS": "Tata Consultancy Services",
    "RELIANCE": "Reliance Industries",
    "HINDUSTAN UNILEVER": "Hindustan Unilever",
    "SUN PHARMA": "Sun Pharmaceutical"
}

In [7]:
import requests
API_KEY = os.getenv("NEWS_API_KEY")

# Use an f-string to inject the variable value into the string
url = (f'https://newsapi.org/v2/everything?'
       f'q=Apple&'
       f'from=2026-03-20&'
       f'sortBy=popularity&'
       f'apiKey={API_KEY}')

response = requests.get(url)
data = response.json()

print(data)

{'status': 'ok', 'totalResults': 449, 'articles': [{'source': {'id': None, 'name': 'NPR'}, 'author': 'Mika Ellison', 'title': '10 tried-and-true methods to stay off your phone, according to our readers', 'description': 'We asked our audience to share the creative ways they limit their own phone use. They range from the practical (keep your phone in another room) to the creative (pair your phone with a fun paperback).', 'url': 'https://www.npr.org/2026/03/20/nx-s1-5752170/clever-effective-ways-to-stay-off-your-phone', 'urlToImage': 'https://media.npr.org/assets/img/2020/04/29/lk_screenlifebalance_1_2_wide-b0c1121ae1134275754f716898e3cb220c315007.jpg?s=1400&c=85&f=jpeg', 'publishedAt': '2026-03-20T09:00:00Z', 'content': "How do you creatively limit your phone use?\r\nWe asked NPR's audience this question last week in a story about how to resist the urge\r\n to keep checking your phone. Experts shared practical tips, like… [+4489 chars]"}, {'source': {'id': 'business-insider', 'name': 'Bu

In [ ]:
import os
from newsapi import NewsApiClient

# 1. SETUP: Replace 'YOUR_ACTUAL_API_KEY' with your key from newsapi.org
# Or ensure your environment variable NEWS_API_KEY is set.
API_KEY = os.getenv("NEWS_API_KEY")

# 2. INITIALIZE the client


def fetch_news(company , API_KEY):
    try:
        # Fetch news
        newsapi = NewsApiClient(api_key=API_KEY)
        response = newsapi.get_everything(
            q=company,
            language="en",
            sort_by="publishedAt",
            page_size=5  # Starting small for testing
        )
        
        # Extract titles safely
        if response and response.get("status") == "ok":
            articles = response.get("articles", [])
            return [a["title"] for a in articles if a.get("title")]
            
        return []
    except Exception as e:
        print(f"Error fetching {company}: {e}")
        return []


        

In [37]:
import yfinance as yf
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import newsapi
from dotenv import load_dotenv
import os
load_dotenv()
def analyze_sentiment_with_langchain(news_text,ticker, stock_name, api_key):
    """Uses LangChain and Gemini to summarize news and determine sentiment."""
    # 1. Fetch News
    #news_text = fetch_news(ticker)
    
    if "No recent news found" in news_text or "Error" in news_text:
        return f"Could not retrieve news for analysis. Details: {news_text}"

    try:
        # 2. Initialize Gemini LLM
        llm = ChatGoogleGenerativeAI(
            temperature=0.2, 
            model="gemini-2.5-flash", 
            api_key=api_key
        )

        # 3. Create LangChain Prompt
        template = """
        You are an expert financial AI assistant. Read the following recent news headlines for {stock_name} ({ticker}).
        
        Recent News:
        {news}
        
        Task:
        1. Provide a 2-3 sentence summary of the current market narrative around this stock.
        2. Classify the overall sentiment as BULLISH, BEARISH, or NEUTRAL.
        3. Identify any potential risks or catalysts mentioned in the headlines.
        
        Format the output cleanly using Markdown.
        """
        
        prompt = PromptTemplate(
            input_variables=["stock_name", "ticker", "news"],
            template=template
        )
        
        # 4. Execute Chain
        chain = prompt | llm
        response = chain.invoke({
            "stock_name": stock_name, 
            "ticker": ticker, 
            "news": news_text
        })
        
        return response.content
        
    except Exception as e:
        return f"Generative AI Error: Please verify your Google API Key and try again. Error details: {e}"

In [28]:
news_text = fetch_news("HDFC Bank")


In [35]:
news_text = " /n ".join(news_text)

In [38]:
ticker = "TCS.NS"
stock_name = "Tata Consultancy Services"
google_key = os.getenv("GOOGLE_API_KEY")
report = analyze_sentiment_with_langchain(news_text,ticker , stock_name, google_key)

In [39]:
report

'Here\'s an analysis of the current market narrative for TCS.NS based on the provided headlines:\n\n### Market Narrative Summary\n\nThe broader market, including the Sensex and Nifty, experienced a climb, primarily attributed to an "IT rebound." This indicates a positive shift in investor sentiment towards the technology sector, suggesting a favorable environment for IT services companies like Tata Consultancy Services.\n\n### Overall Sentiment\n\n**BULLISH**\n\n### Potential Risks or Catalysts\n\n*   **Catalyst:** An explicit "IT rebound" is mentioned as a key driver for market gains, which is a positive catalyst for IT stocks like TCS.\n*   **Potential Risk:** The rupee slipping to 93.71/USD could be a minor concern, potentially signaling broader economic pressures, although a weaker rupee can also benefit IT exporters by increasing the value of their dollar-denominated revenues.'